# Financial GenAI Chatbot - RAG Prototyping

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) QA Chain using Langchain's latest syntax, HuggingFace Embeddings, and the Groq API (as well as Zephyr locally via pipeline).


In [ ]:
!pip install -q langchain langchain-groq langchain-huggingface chromadb sentence-transformers pandas


In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Set API Keys
os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY'  # Replace with your actual Groq API Key


## 1. Load Data


In [ ]:
bank = pd.read_csv('BankFAQs.csv')
bank['content'] = bank.apply(lambda row: f"Question: {row['Question']}\nAnswer: {row['Answer']}", axis=1)
bank.head()


In [ ]:
from langchain.docstore.document import Document

# Prepare documents for LangChain
documents = []
for _, row in bank.iterrows():
    documents.append(Document(page_content=row['content'], metadata={'class': row.get('Class', 'unknown')}))


## 2. Generate Embeddings & Load into ChromaDB


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Using local HuggingFace embeddings
hg_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

persist_directory = './chroma_db/'
langchain_chroma = Chroma.from_documents(
    documents=documents,
    collection_name="chatbot_finance",
    embedding=hg_embeddings,
    persist_directory=persist_directory
)


## 3. Initialize LLM (Groq)


In [ ]:
from langchain_groq import ChatGroq

# Using Mixtral via Groq (Fastest and free tier open-source inference)
llm = ChatGroq(temperature=0, model_name="mixtral-8x7b-32768")

# --- OPTIONAL: HUGGINGFACE LOCAL ZEPHYR PIPELINE --- 
# Uncomment the below code to use Zephyr locally. Note: Requires significant VRAM/RAM.
# from langchain_huggingface import HuggingFacePipeline
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# import torch
# model_id = 'HuggingFaceH4/zephyr-7b-beta'
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(model_id, load_in_4bit=True, device_map='auto')
# query_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)
# llm = HuggingFacePipeline(pipeline=query_pipeline)


## 4. Build RAG Chain


In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

template = """
You are a Finance QNA Expert. Analyze the Query and Respond to Customer with a suitable answer.
If you don't know the answer based on the context provided, just say "Sorry, I don't know."

Context: {context}
Question: {input}
Answer: """

PROMPT = PromptTemplate.from_template(template)

retriever = langchain_chroma.as_retriever(search_kwargs={"k": 3})

document_chain = create_stuff_documents_chain(llm, PROMPT)
qa_chain = create_retrieval_chain(retriever, document_chain)


In [ ]:
def chat_with_rag():
    print("Welcome to the GenAI Financial Chatbot. Type 'exit' to end the conversation.")
    while True:
        query = input("You: ")
        if query.lower() in ["exit", "quit"]:
            break
        try:
            result = qa_chain.invoke({"input": query})
            print(f"Chatbot: {result['answer']}\n")
        except Exception as e:
            print(f"Error encountered: {e}")

# Run the chat in the notebook
# chat_with_rag()
